In [1]:
# 모듈 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import classification_report

In [2]:
# 데이터 로딩
df = pd.read_csv('/content/drive/MyDrive/KDThome/santander-customer-transaction-prediction/train.csv')

In [3]:
# 결측치 전체 데이터에서 하나도 없음!
df.isnull().sum().sum()

df_1 = df.drop('ID_code', axis=1, inplace=False)

In [ ]:
# feature은 corr 기반 상위 20개와 타겟별 평균 차이 기반 상위 20개 (겹치는거는 하나만) feature로 변수 정리

# 1. 타겟별 평균 차이 기반 상위 20개
mean_diff = df_1.groupby('target').mean()
display(mean_diff)

diff = (mean_diff.loc[1] - mean_diff.loc[0]).abs()
top100_diff = diff.sort_values(ascending=False).head(100).index.tolist()

print(top100_diff)

,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
target,,,,,,,,,,,,,,,,,,,,,
0,10.626681,-1.695770,10.665876,6.788979,11.072412,-5.146736,5.389620,16.549306,0.262347,7.584907,...,3.149130,7.39080,1.949017,3.355403,18.017716,-0.155601,2.260297,8.919032,15.924058,-3.415273
1,11.156418,-1.017613,11.156633,6.864113,11.131337,-4.336522,5.581966,16.514917,0.479432,7.409064,...,3.998064,7.86456,1.738266,3.120260,17.779568,-0.021130,2.688583,8.810815,15.393283,-2.532243


['var_139', 'var_76', 'var_149', 'var_21', 'var_184', 'var_174', 'var_45', 'var_80', 'var_40', 'var_90', 'var_26', 'var_48', 'var_118', 'var_18', 'var_172', 'var_67', 'var_70', 'var_107', 'var_86', 'var_147', 'var_44', 'var_74', 'var_165', 'var_199', 'var_13', 'var_190', 'var_173', 'var_123', 'var_110', 'var_5', 'var_137', 'var_49', 'var_167', 'var_75', 'var_154', 'var_164', 'var_122', 'var_109', 'var_155', 'var_135', 'var_51', 'var_170', 'var_1', 'var_87', 'var_141', 'var_92', 'var_97', 'var_33', 'var_82', 'var_35', 'var_81', 'var_157', 'var_22', 'var_187', 'var_83', 'var_178', 'var_163', 'var_180', 'var_146', 'var_198', 'var_0', 'var_102', 'var_2', 'var_191', 'var_89', 'var_179', 'var_52', 'var_11', 'var_188', 'var_54', 'var_120', 'var_115', 'var_119', 'var_196', 'var_94', 'var_56', 'var_127', 'var_145', 'var_36', 'var_151', 'var_99', 'var_20', 'var_142', 'var_24', 'var_134', 'var_58', 'var_55', 'var_186', 'var_177', 'var_78', 'var_85', 'var_47', 'var_19', 'var_128', 'var_61', 'var_1

In [ ]:
# 2. 상관계수 기반 상위 20개
corrs = df_1.corr()['target'].abs().sort_values(ascending=False)
display(corrs.head(100))
top20_corr = corrs.head(101).index.tolist()
print(top20_corr)


,target
target,1.000000
var_81,0.080917
var_139,0.074080
var_12,0.069489
var_6,0.066731
...,...
var_137,0.027190
var_128,0.026909
var_70,0.026748
var_111,0.026686


['target', 'var_81', 'var_139', 'var_12', 'var_6', 'var_110', 'var_146', 'var_53', 'var_26', 'var_76', 'var_174', 'var_22', 'var_21', 'var_99', 'var_166', 'var_80', 'var_190', 'var_2', 'var_165', 'var_13', 'var_148', 'var_133', 'var_198', 'var_34', 'var_0', 'var_1', 'var_115', 'var_179', 'var_109', 'var_40', 'var_44', 'var_169', 'var_184', 'var_78', 'var_170', 'var_149', 'var_191', 'var_94', 'var_92', 'var_154', 'var_108', 'var_67', 'var_33', 'var_18', 'var_192', 'var_9', 'var_122', 'var_173', 'var_164', 'var_118', 'var_123', 'var_147', 'var_91', 'var_107', 'var_121', 'var_89', 'var_86', 'var_127', 'var_95', 'var_36', 'var_75', 'var_172', 'var_155', 'var_177', 'var_35', 'var_87', 'var_197', 'var_93', 'var_56', 'var_188', 'var_71', 'var_106', 'var_162', 'var_157', 'var_131', 'var_48', 'var_163', 'var_180', 'var_5', 'var_119', 'var_145', 'var_167', 'var_49', 'var_32', 'var_186', 'var_130', 'var_141', 'var_90', 'var_43', 'var_24', 'var_195', 'var_125', 'var_135', 'var_52', 'var_151', 'var

In [ ]:
# 3. 두 방식의 feature 합치기
# 두 방식 조합 (중복 제거)
combined_features = list(set(top100_diff + top20_corr))
#display(combined_features)

df_2 = df_1[combined_features]
display(df_2.head())
df_2.info()

,var_6,var_20,var_47,var_75,var_9,var_19,var_139,var_24,var_155,var_157,...,var_109,var_53,var_85,var_49,var_78,var_55,var_128,var_44,var_127,var_130
0,5.1187,10.5350,-14.2136,18.3816,5.7470,30.7133,15.6599,14.3831,1.6573,-13.1324,...,24.3627,5.1736,21.4669,5.3253,6.5199,14.8322,-1.9245,11.6418,-0.7338,12.8287
1,5.6208,3.4287,0.1948,7.0529,8.0851,28.5708,16.1622,6.9779,0.1898,-9.6953,...,13.0858,6.6885,13.7867,25.7037,5.5075,18.5995,0.8194,1.2444,2.4354,12.4205
2,6.9427,17.7559,-5.7864,19.4465,5.9525,20.4775,8.6674,5.6777,0.5778,-1.7624,...,20.3882,6.4059,19.0773,6.8874,6.3191,6.2846,-0.9479,4.1006,-2.5511,11.5419
3,5.8428,20.3010,-35.1659,15.4235,8.2450,13.7257,8.9821,12.1354,-10.9370,-1.2155,...,14.4135,5.2091,17.9762,8.3838,4.0806,12.3972,3.5974,8.0485,-1.3683,14.3003
4,5.9405,21.4246,0.0444,23.3521,7.6784,11.3152,13.9547,14.2080,10.6101,-12.6068,...,28.2749,5.7555,14.5265,14.4268,7.1734,14.1482,5.6518,6.9087,7.0642,11.4266


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Columns: 128 entries, var_6 to var_130
dtypes: float64(127), int64(1)
memory usage: 195.3 MB


In [ ]:
from sklearn.model_selection import train_test_split

# DataFrame 형태 유지
X_combined_df = df_3.drop('target', axis=1)
t_combined_data = df_3['target'].values

# 그 이후 train/test split
x_train, x_test, t_train, t_test = train_test_split(
    X_combined_df.values, t_combined_data,
    test_size=0.2,
    stratify=t_combined_data,
)

# 3. RobustScaler 적용
scaler = RobustScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


# 오버샘플링
sm = SMOTE(random_state=42)
x_train_sm, t_train_sm = sm.fit_resample(x_train, t_train)

# 모델 학습
model = XGBClassifier(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.2,
)
model.fit(x_train_sm, t_train_sm)

# 예측 및 리포트
t_pred = model.predict(x_test)
print(f"\n🔸 모델 성능 🔸")
print(classification_report(t_test, t_pred, digits=3))




🔸 모델 성능 🔸
              precision    recall  f1-score   support

           0      0.933     0.879     0.906     35980
           1      0.289     0.439     0.349      4020

    accuracy                          0.835     40000
   macro avg      0.611     0.659     0.627     40000
weighted avg      0.869     0.835     0.850     40000



In [ ]:

# ✅ feature 중요도 추출
importances = model.feature_importances_
feature_names = X_combined_df.columns

importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
importance_df = importance_df.sort_values(by='importance', ascending=False).reset_index(drop=True)

# 상위 n개 출력
print(importance_df.head(20))

In [ ]:
# 중요도 기반 top 20 feature만 추출
top_features_d = importance_df['feature'].head(100).tolist()

# 데이터셋 구성
df_d = df_2[top_features_d + ['target']]

# 학습/테스트 분리
X = df_d.drop('target', axis=1)
y = df_d['target']

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 오버샘플링
sm = SMOTE(random_state=42)
x_train_sm, y_train_sm = sm.fit_resample(x_train, y_train)

# 모델 학습
model_d = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_d.fit(x_train_sm, y_train_sm)

# 예측 및 평가
y_pred = model_d.predict(x_test)
print("🔸 [D] 중요도 기반 top20 feature 기준 모델 성능 🔸")
print(classification_report(y_test, y_pred, digits=3))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:35:11] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


🔸 [D] 중요도 기반 top20 feature 기준 모델 성능 🔸
              precision    recall  f1-score   support

           0      0.927     0.870     0.898     35980
           1      0.251     0.389     0.305      4020

    accuracy                          0.822     40000
   macro avg      0.589     0.630     0.601     40000
weighted avg      0.859     0.822     0.838     40000



In [ ]:
# 평균차 + 상관계수 (C) + 중요도 기반 (D) 조합 feature로 학습 & 평가

feature_c = [
    'var_184', 'var_40', 'var_90', 'var_26', 'var_48', 'var_118', 'var_18', 'var_67', 'var_70', 'var_147',
    'var_74', 'var_199', 'var_190', 'var_173', 'var_110', 'var_5', 'var_137', 'var_49', 'var_167', 'var_164'
]

feature_d = [
    'var_6', 'var_53', 'var_12', 'var_166', 'var_81', 'var_13', 'var_2', 'var_99', 'var_22', 'var_148',
    'var_190', 'var_139', 'var_146', 'var_110', 'var_174', 'var_26', 'var_165', 'var_76', 'var_149', 'var_67'
]

feature_cd = list(set(feature_c + feature_d))
print(feature_cd)

['var_184', 'var_90', 'var_22', 'var_5', 'var_146', 'var_6', 'var_118', 'var_12', 'var_13', 'var_2', 'var_139', 'var_74', 'var_18', 'var_148', 'var_149', 'var_174', 'var_190', 'var_40', 'var_167', 'var_166', 'var_165', 'var_76', 'var_48', 'var_81', 'var_99', 'var_67', 'var_70', 'var_147', 'var_26', 'var_53', 'var_164', 'var_137', 'var_49', 'var_173', 'var_199', 'var_110']


In [ ]:
# 데이터셋 구성
df_3 = df_1[[
    'target','var_184', 'var_90', 'var_22', 'var_5', 'var_146', 'var_6', 'var_118', 'var_12', 'var_13', 'var_2', 'var_139', 'var_74', 'var_18', 'var_148', 'var_149', 'var_174', 'var_190', 'var_40', 'var_167', 'var_166', 'var_165', 'var_76', 'var_48', 'var_81', 'var_99', 'var_67', 'var_70', 'var_147', 'var_26', 'var_53', 'var_164', 'var_137', 'var_49', 'var_173', 'var_199', 'var_110'
    ]]

display(df_3.head())

,target,var_184,var_90,var_22,var_5,var_146,var_6,var_118,var_12,var_13,...,var_70,var_147,var_26,var_53,var_164,var_137,var_49,var_173,var_199,var_110
0,0,25.8398,-21.4494,2.5791,-9.2834,11.5659,5.1187,-13.4221,14.0137,0.5745,...,21.6374,-16.4727,-5.1488,5.1736,6.6760,31.4045,5.3253,3.1531,-1.0914,2.0323
1,0,22.5441,0.4768,8.5524,7.0433,8.9231,5.6208,-11.5100,14.0239,8.4135,...,40.5632,11.7700,-11.7684,6.6885,-5.0121,18.1577,25.7037,5.5134,1.9518,6.6203
2,0,23.0866,-22.4038,1.2145,-9.0837,11.4934,6.9427,-17.2738,14.1929,7.3124,...,2.3612,1.7624,-7.9940,6.4059,-1.0410,15.5827,6.8874,-5.8234,0.3965,3.2304
3,0,-0.4639,-7.5866,6.8202,-1.8361,10.4994,5.8428,3.5732,13.8463,11.9704,...,4.0462,4.1622,0.8135,5.2091,-11.0882,24.6065,8.3838,11.7134,-8.9996,4.2827
4,0,11.8503,-39.7997,10.1102,2.4486,11.5670,5.9405,13.8224,13.8481,7.8895,...,40.1236,-12.7047,3.1736,5.7555,6.5769,25.8128,14.4268,2.3705,-8.8104,-0.1937


In [ ]:
# train/test split
X = df_3.drop('target', axis=1, inplace=False)
y = df_3['target'].values

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 오버샘플링
sm = SMOTE(random_state=42)
x_train_sm, y_train_sm = sm.fit_resample(x_train, y_train)

# 모델 학습
model_e = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_e.fit(x_train_sm, y_train_sm)

# 예측 및 평가
y_pred = model_e.predict(x_test)
print("🔸 [E] 평균차 + 상관계수 + 중요도 기반 조합 모델 성능 🔸")
print(classification_report(y_test, y_pred, digits=3))

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [03:55:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


🔸 [E] 평균차 + 상관계수 + 중요도 기반 조합 모델 성능 🔸
              precision    recall  f1-score   support

           0      0.923     0.765     0.837     35980
           1      0.171     0.432     0.245      4020

    accuracy                          0.732     40000
   macro avg      0.547     0.599     0.541     40000
weighted avg      0.848     0.732     0.777     40000



In [ ]:
# E타입 feature 중요도 추출
importances_e = model_e.feature_importances_
feature_names_e = X.columns

importance_dfe = pd.DataFrame({'feature': feature_names_e, 'importance': importances_e})
importance_dfe = importance_dfe.sort_values(by='importance', ascending=False).reset_index(drop=True)

# 상위 n개 출력
print(importance_dfe.head(25))

    feature  importance
0     var_6    0.051043
1    var_81    0.041385
2    var_53    0.039382
3    var_12    0.037586
4    var_99    0.035328
5    var_13    0.034829
6   var_148    0.034540
7   var_166    0.034441
8    var_22    0.033882
9     var_2    0.033818
10  var_110    0.033591
11  var_146    0.033082
12  var_139    0.032572
13  var_173    0.032484
14  var_165    0.030869
15  var_190    0.030691
16  var_174    0.028123
17   var_26    0.027615
18  var_164    0.026365
19   var_67    0.025413
20   var_76    0.024109
21   var_18    0.023266
22  var_184    0.022968
23  var_149    0.022451
24  var_118    0.021768


In [ ]:
# 중요도 상위 25개 feature 추출
top_25_features = importance_dfe['feature'].head(25).tolist()

# X, y 구성
X_top25 = df_3[top_25_features]
y_top25 = df_3['target']

# 학습/테스트 분리
x_train, x_test, y_train, y_test = train_test_split(
    X_top25, y_top25, test_size=0.2, stratify=y_top25, random_state=42
)

# SMOTE 오버샘플링
sm = SMOTE(random_state=42)
x_train_sm, y_train_sm = sm.fit_resample(x_train, y_train)

# 모델 학습
model_top25 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_top25.fit(x_train_sm, y_train_sm)

# 예측 및 성능 평가
y_pred = model_top25.predict(x_test)
print("🔸 [E-Top25] 중요도 상위 25개 feature 기준 모델 성능 🔸")
print(classification_report(y_test, y_pred, digits=3))

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:02:42] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


🔸 [E-Top25] 중요도 상위 25개 feature 기준 모델 성능 🔸
              precision    recall  f1-score   support

           0      0.927     0.739     0.822     35980
           1      0.171     0.483     0.253      4020

    accuracy                          0.713     40000
   macro avg      0.549     0.611     0.538     40000
weighted avg      0.851     0.713     0.765     40000



In [ ]:
print(top_25_features)

['var_6', 'var_81', 'var_53', 'var_12', 'var_99', 'var_13', 'var_148', 'var_166', 'var_22', 'var_2', 'var_110', 'var_146', 'var_139', 'var_173', 'var_165', 'var_190', 'var_174', 'var_26', 'var_164', 'var_67', 'var_76', 'var_18', 'var_184', 'var_149', 'var_118']


In [ ]:
# 컬럼 25개로 추렸으니 이제 데이터 전처리 고고

df_4 = df_1[[
    'target','var_6', 'var_81', 'var_53', 'var_12', 'var_99', 'var_13', 'var_148', 'var_166', 'var_22', 'var_2', 'var_110', 'var_146', 'var_139', 'var_173', 'var_165', 'var_190', 'var_174', 'var_26', 'var_164', 'var_67', 'var_76', 'var_18', 'var_184', 'var_149', 'var_118'
]]
display(df_4.head())
df_4.info()


,target,var_6,var_81,var_53,var_12,var_99,var_13,var_148,var_166,var_22,...,var_190,var_174,var_26,var_164,var_67,var_76,var_18,var_184,var_149,var_118
0,0,5.1187,13.8372,5.1736,14.0137,-3.4132,0.5745,4.0288,2.7004,2.5791,...,4.4354,18.5618,-5.1488,6.6760,22.4321,-2.3440,4.2840,25.8398,17.9244,-13.4221
1,0,5.6208,18.1782,6.6885,14.0239,0.6939,8.4135,4.2578,3.2003,8.5524,...,7.6421,30.2645,-11.7684,-5.0121,7.9344,3.2709,7.8000,22.5441,-4.4223,-11.5100
2,0,6.9427,15.7811,6.4059,14.1929,-0.0269,7.3124,4.0714,3.2790,1.2145,...,2.9057,25.6820,-7.9940,-1.0410,9.8565,4.5048,4.7011,23.0866,-1.2681,-17.2738
3,0,5.8428,10.5404,5.2091,13.8463,1.9480,11.9704,3.7613,2.5881,6.8202,...,4.4666,14.7483,0.8135,-11.0882,23.6143,11.6875,15.9426,-0.4639,2.3701,3.5732
4,0,5.9405,13.3317,5.7555,13.8481,0.6715,7.8895,3.7574,3.2304,10.1102,...,-1.4905,18.4685,3.1736,6.5769,1.6184,1.0273,6.5263,11.8503,9.9110,13.8224


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 26 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   target   200000 non-null  int64  
 1   var_6    200000 non-null  float64
 2   var_81   200000 non-null  float64
 3   var_53   200000 non-null  float64
 4   var_12   200000 non-null  float64
 5   var_99   200000 non-null  float64
 6   var_13   200000 non-null  float64
 7   var_148  200000 non-null  float64
 8   var_166  200000 non-null  float64
 9   var_22   200000 non-null  float64
 10  var_2    200000 non-null  float64
 11  var_110  200000 non-null  float64
 12  var_146  200000 non-null  float64
 13  var_139  200000 non-null  float64
 14  var_173  200000 non-null  float64
 15  var_165  200000 non-null  float64
 16  var_190  200000 non-null  float64
 17  var_174  200000 non-null  float64
 18  var_26   200000 non-null  float64
 19  var_164  200000 non-null  float64
 20  var_67   200000 non-null  

In [ ]:
# 데이터 전처리
# 결측치는 없고~
# 이상치 확인 고고
print(df_4.shape)
print(df_4['target'].value_counts())


(200000, 26)
target
0    179902
1     20098
Name: count, dtype: int64


In [ ]:
# 이상치 변환 함수 (IQR 클리핑)
def clip_outliers_iqr(df):
  df_clipped = df.copy()
  for col in df.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    df_clipped[col] = df[col].clip(lower, upper)

  return df_clipped

target_out = df_4.drop('target', axis=1, inplace=False)
df_5 = clip_outliers_iqr(target_out)

df_6 = pd.concat([df_5,df_4['target']], axis=1)

print(df_6.shape)
print(df_6['target'].value_counts())

(200000, 26)
target
0    179902
1     20098
Name: count, dtype: int64


In [ ]:
# 데이터 분리
x_data = df_6.drop('target', axis=1, inplace=False).values
t_data = df_6['target'].values

x_train, x_test, y_train, y_test = train_test_split(
    x_data, t_data, test_size=0.1, stratify=t_data
)

In [ ]:
# 오버샘플링
sm = SMOTE(random_state=42)
x_train_sm, t_train_sm = sm.fit_resample(x_data, t_data)

In [ ]:
# 3. 정규화
scaler = StandardScaler()
x_train_norm = scaler.fit_transform(x_train_sm)
x_test_norm = scaler.transform(x_test)

# 모델 학습
xgb_model = XGBClassifier(n_estimators=50,
                          max_depth=3,
                          learning_rate=0.1,
                          random_state=42
                          )
xgb_model.fit(x_train_norm, t_train_sm)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=50, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [ ]:
# 5. 예측 및 평가
y_pred = xgb_model.predict(x_test_norm)
print("🔸 [F] 평균차 + 상관계수 + 중요도 기반 조합 + 이상치처리 모델 성능 🔸")
print(classification_report(y_test, y_pred, digits=3))

🔸 [F] 평균차 + 상관계수 + 중요도 기반 조합 + 이상치처리 모델 성능 🔸
              precision    recall  f1-score   support

           0      0.933     0.684     0.789     17990
           1      0.166     0.562     0.256      2010

    accuracy                          0.672     20000
   macro avg      0.550     0.623     0.523     20000
weighted avg      0.856     0.672     0.736     20000



In [ ]:
df_e['var6_minus_12'] = df_e['var_6'] - df_e['var_12']
df_e['var81_div_13'] = df_e['var_81'] / (df_e['var_13'] + 1e-6)

In [ ]:
# 평가용 데이터 불러오기
test = pd.read_csv('/content/drive/MyDrive/KDThome/santander-customer-transaction-prediction/test.csv')